# NB09 — Identifying Research Gaps from a Literature Body

**Module:** 19 — Research Seminar and Paper Reading  
**Estimated time:** 2 hours  

---

## Motivation

Identifying a genuine research gap is the hardest skill in science. It requires knowing the field well enough to see what is missing, distinguishing between "no one has done this" and "no one has done this because it does not matter," and framing the gap as a question that is both open and answerable. This notebook builds a systematic framework for doing this rather than relying on serendipity.

## 1. Gap Identification Framework

Four dimensions of research gaps in computational biology:

| Gap type | Question | Example |
|----------|----------|--------|
| **Technical** | What methods exist, and where do they fail? | scRNA-seq integration tools break when batch effects exceed 40% variance; no method handles spatial+temporal batches simultaneously |
| **Biological** | What biological questions are unanswered? | What proportion of cell-type-specific gene regulation occurs at the post-transcriptional level? |
| **Dataset** | Where is data missing? | No longitudinal scRNA-seq dataset tracks the same cells across disease progression |
| **Validation** | Where are results not independently validated? | AlphaFold2's performance on intrinsically disordered proteins is rarely benchmarked |
| **Application** | Where could a known method be applied to a new domain? | Protein language models have not been applied to viral evolution at pandemic scale |

## 2. Gap Signal Language in Abstracts

Papers signal gaps using specific sentence patterns. Learning to spot them is a fast way to build a gap map:

- `"However, ..."` — contrast with prior work; often introduces a gap
- `"Despite ..., no method ..."` — explicit gap statement
- `"...remains unclear"` — open biological question
- `"...is limited by"` — technical gap
- `"Future work should ..."` — authors name their own gap
- `"...fails to account for"` — limitation claim
- `"Although ..., ..."` — concessive structure; second clause often identifies a gap

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
from collections import defaultdict

# Import LiteratureMap from NB07 (replicated here for self-containment)

@dataclass
class LiteraturePaper:
    title: str
    authors: str
    year: int
    topic: str
    method: str
    organism: str
    dataset_size: int
    n_citations: int = 0


def identify_gap_sentences(abstract_text: str) -> List[str]:
    """Return list of sentences containing gap-signal words.

    Gap signals: however, despite, fail, limitation, unclear, remains,
    although, yet, limited by, future work.
    """
    GAP_SIGNALS = [
        r'\bhowever\b', r'\bdespite\b', r'\bfail(s|ed|ing)?\b',
        r'\blimitation(s)?\b', r'\bremains? unclear\b', r'\bremains? unknown\b',
        r'\balthough\b', r'\byet\b', r'\blimited by\b', r'\bfuture work\b',
        r'\bchallenging\b', r'\bdifficult to\b', r'\bunsolved\b',
        r'\bnot been\b', r'\bno method\b', r'\blacks?\b',
    ]
    sentences = re.split(r'(?<=[.!?])\s+', abstract_text.strip())
    gap_sentences = []
    for sentence in sentences:
        if any(re.search(sig, sentence, re.IGNORECASE) for sig in GAP_SIGNALS):
            gap_sentences.append(sentence.strip())
    return gap_sentences


@dataclass
class ResearchGap:
    gap_type: str      # technical, biological, dataset, validation, application
    description: str
    opportunity_score: float   # 0-10: how important/tractable is this gap?
    difficulty: float          # 0-10: how hard to address?
    impact: float              # 0-10: potential field impact
    source_papers: List[str] = field(default_factory=list)


def compute_gap_opportunity(gap: ResearchGap) -> float:
    """Score = impact * (1 / (1 + difficulty/10)) — high impact, tractable gaps score highest."""
    tractability = 1 / (1 + gap.difficulty / 10)
    return round(gap.impact * tractability, 2)


# Test gap sentence detector on synthetic abstracts
test_abstracts = [
    """Single-cell RNA-seq has transformed our understanding of cell types.
    However, current integration methods fail to handle spatial and temporal
    batch effects simultaneously. Although several tools exist, none accounts
    for cell state transitions across time points. This remains a key limitation
    for longitudinal studies.""",

    """AlphaFold2 has achieved near-experimental accuracy for folded proteins.
    Despite this, performance on intrinsically disordered regions remains unclear.
    Future work should address this limitation, as disordered regions play
    important roles in signaling.""",

    """We present a novel aligner that achieves state-of-the-art speed.
    The method is benchmarked on three datasets and achieves top performance.
    All code is available at github.com/lab/aligner.""",
]

for i, abstract in enumerate(test_abstracts):
    gaps = identify_gap_sentences(abstract)
    print(f"Abstract {i+1}: {len(gaps)} gap sentences")
    for g in gaps:
        print(f"  -> {g[:80]}..." if len(g) > 80 else f"  -> {g}")

In [ ]:
# Build a research gap portfolio for a synthetic sub-field
gaps = [
    ResearchGap('technical', 'No scRNA-seq integration handles spatial+temporal batches',
                opportunity_score=8.2, difficulty=7.0, impact=9.0,
                source_papers=['butler2018', 'korsunsky2019']),
    ResearchGap('biological', 'Proportion of cell-type regulation at post-transcriptional level unknown',
                opportunity_score=7.5, difficulty=8.0, impact=9.5,
                source_papers=['love2014', 'wolf2018']),
    ResearchGap('dataset', 'No longitudinal scRNA-seq dataset tracking cells across disease progression',
                opportunity_score=6.0, difficulty=9.0, impact=8.0,
                source_papers=['butler2018']),
    ResearchGap('validation', 'AlphaFold2 performance on intrinsically disordered proteins not benchmarked',
                opportunity_score=7.0, difficulty=5.0, impact=7.5,
                source_papers=['jumper2021']),
    ResearchGap('application', 'Protein language models not applied to viral evolution at pandemic scale',
                opportunity_score=8.5, difficulty=6.0, impact=9.0,
                source_papers=['jumper2021']),
    ResearchGap('technical', 'Clustering resolution sensitivity not systematically characterized',
                opportunity_score=5.0, difficulty=3.0, impact=6.0,
                source_papers=['wolf2018']),
    ResearchGap('biological', 'Cell state vs. cell type boundary poorly defined computationally',
                opportunity_score=6.5, difficulty=8.5, impact=8.5,
                source_papers=['butler2018', 'wolf2018']),
    ResearchGap('dataset', 'Rare cell types (< 0.1%) systematically underrepresented in benchmarks',
                opportunity_score=7.0, difficulty=5.5, impact=7.0,
                source_papers=['korsunsky2019']),
]

for gap in gaps:
    gap.opportunity_score = compute_gap_opportunity(gap)

gaps_sorted = sorted(gaps, key=lambda g: g.opportunity_score, reverse=True)
print("Research gap opportunity ranking:")
for i, g in enumerate(gaps_sorted):
    print(f"  {i+1}. [{g.gap_type}] {g.description[:60]}... score={g.opportunity_score}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Research Gap Analysis — Computational Transcriptomics', fontsize=12, fontweight='bold')

# Panel 1: Gap radar chart (5-axis)
gap_type_counts = defaultdict(int)
gap_type_scores = defaultdict(float)
for g in gaps:
    gap_type_counts[g.gap_type] += 1
    gap_type_scores[g.gap_type] += g.opportunity_score

gap_types = ['technical', 'biological', 'dataset', 'validation', 'application']
radar_values = [gap_type_scores.get(t, 0) for t in gap_types]
radar_max = max(radar_values) if radar_values else 1
radar_norm = [v / radar_max for v in radar_values]

N = len(gap_types)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]
radar_norm += radar_norm[:1]

ax1 = fig.add_subplot(131, polar=True)
ax1.plot(angles, radar_norm, 'o-', linewidth=2, color='steelblue')
ax1.fill(angles, radar_norm, alpha=0.2, color='steelblue')
ax1.set_xticks(angles[:-1])
ax1.set_xticklabels([t.capitalize() for t in gap_types], size=9)
ax1.set_ylim(0, 1)
ax1.set_yticks([0.25, 0.5, 0.75, 1.0])
ax1.set_yticklabels(['25%', '50%', '75%', '100%'], size=7)
ax1.set_title('Gap Radar\n(total opportunity by type)', pad=20)

# Panel 2: Opportunity matrix (difficulty vs impact)
ax2 = axes[1]
type_colors = {'technical': '#2196F3', 'biological': '#4CAF50', 'dataset': '#FF9800',
               'validation': '#9C27B0', 'application': '#E53935'}
for g in gaps:
    color = type_colors.get(g.gap_type, 'gray')
    ax2.scatter(g.difficulty, g.impact, s=g.opportunity_score * 30 + 50,
                c=color, alpha=0.8, edgecolors='white', linewidth=1)
    label_text = g.description[:25] + '...'
    ax2.annotate(label_text, (g.difficulty, g.impact),
                 textcoords='offset points', xytext=(5, 3), fontsize=6)

ax2.set_xlabel('Difficulty (0=easy, 10=hard)')
ax2.set_ylabel('Potential impact (0=low, 10=high)')
ax2.set_title('Opportunity Matrix\n(size = opportunity score)')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 11)
ax2.set_ylim(0, 11)

# Add quadrant labels
ax2.axvline(5, color='gray', linestyle=':', alpha=0.5)
ax2.axhline(5, color='gray', linestyle=':', alpha=0.5)
ax2.text(1, 9.5, 'High impact\nLow difficulty\n(prime targets)', fontsize=7, color='green', alpha=0.7)
ax2.text(7, 9.5, 'High impact\nHigh difficulty\n(moonshots)', fontsize=7, color='orange', alpha=0.7)

# Panel 3: Ranked bar chart
ax3 = axes[2]
labels = [f"{g.gap_type[:4]}: {g.description[:30]}..." for g in gaps_sorted]
scores = [g.opportunity_score for g in gaps_sorted]
bar_colors = [type_colors.get(g.gap_type, 'gray') for g in gaps_sorted]
y_pos = np.arange(len(labels))
ax3.barh(y_pos, scores, color=bar_colors, alpha=0.85)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(labels, fontsize=7)
ax3.set_xlabel('Opportunity score')
ax3.set_title('Ranked Research Gaps')
ax3.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('research_gaps.png', dpi=150, bbox_inches='tight')
plt.show()

## Exercises

**Exercise 1:** Apply `identify_gap_sentences()` to the abstract of Jumper et al. (2021) AlphaFold2. How many gap sentences does it find? Are they all genuine gaps, or some of them statements of contribution?

**Exercise 2:** Build a `ResearchGap` list for the sub-field you are targeting (from your emerging research interest after Month 3). Score each gap. Which is your prime target?

**Exercise 3:** Write a one-paragraph "research statement" positioning yourself relative to the highest-scored gap from Exercise 2. Use the format: "The field has achieved X. However, Y remains unsolved because Z. I propose to address this by W."

**Exercise 4 (reflection):** What is the difference between "no one has done this" and "no one has done this because it doesn't matter"? How do you tell the difference from the literature alone?

## Quiz

1. Name the five types of research gap.
2. Name three sentence patterns that signal a gap in an abstract.
3. What does the opportunity score formula measure, and why does it balance impact and difficulty?
4. What is a "prime target" gap (high impact + low difficulty), and why is it valuable for a 12-month programme?
5. Why is a dataset gap sometimes harder to address than a technical gap?

## References

- Keshav, S. (2007). How to Read a Paper. ACM SIGCOMM CCR. (Section on identifying contributions)
- Pautasso, M. (2013). Ten simple rules for writing a literature review. PLOS Comp Bio.